In [2]:
import requests
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

# 1. Load the secrets from the .env file
load_dotenv()

# --- CONFIGURATION ---
CLIENT_ID = os.getenv("OPENSKY_CLIENT_ID")
CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET")

# 2. Initialize a properly version-matched Local Spark Session
spark = SparkSession.builder \
    .appName("Aviation_Density_Local") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# 3. Get OAuth2 Access Token
def get_opensky_token(client_id, client_secret):
    token_url = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
    data = {"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret}
    
    response = requests.post(token_url, data=data)
    if response.status_code == 200:
        return response.json().get("access_token")
    else:
        raise Exception(f"Failed to get token: {response.status_code} - {response.text}")

token = get_opensky_token(CLIENT_ID, CLIENT_SECRET)
# 2. Fetch Data
url = "https://opensky-network.org/api/states/all"
headers = {"Authorization": f"Bearer {token}"}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    flight_vectors = response.json().get("states", [])[:2000]
    print(f"✅ Success! Retrieved {len(flight_vectors)} flights.")
else:
    raise Exception(f"❌ API Error {response.status_code}: {response.text}")

# 3. Stringify the Python Data
safe_data = [[str(i) if i is not None else None for i in f] for f in flight_vectors]

# 4. EXPLICIT BRONZE SCHEMA: Force everything to be a String
bronze_schema = StructType([
    StructField("icao24", StringType(), True), StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True), StructField("time_position", StringType(), True),
    StructField("last_contact", StringType(), True), StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True), StructField("baro_altitude", StringType(), True),
    StructField("on_ground", StringType(), True), StructField("velocity", StringType(), True),
    StructField("true_track", StringType(), True), StructField("vertical_rate", StringType(), True),
    StructField("sensors", StringType(), True), StructField("geo_altitude", StringType(), True),
    StructField("squawk", StringType(), True), StructField("spi", StringType(), True),
    StructField("position_source", StringType(), True)
])

# 5. Initialize Spark & Write BRONZE
spark = SparkSession.builder.getOrCreate()
df_bronze = spark.createDataFrame(safe_data, schema=bronze_schema)
df_bronze.write.format("delta").mode("append").save("./data/bronze_flights")
print("✅ Bronze layer saved safely!")

# 6. Transform & Write SILVER
# Now we confidently cast to floats/booleans, knowing PySpark won't crash on the read
df_silver = df_bronze.select(
    col("icao24"),
    col("callsign"),
    col("longitude").cast("float"),      
    col("latitude").cast("float"),
    col("baro_altitude").cast("float"),
    col("velocity").cast("float"),
    col("true_track").cast("float"),
    col("on_ground").cast("boolean")     
).filter(
    (col("on_ground") == False) &          
    (col("latitude").isNotNull()) &        
    (col("longitude").isNotNull()) &
    (col("baro_altitude").isNotNull()) &   
    (col("velocity") > 0)                  
)

df_silver.write.format("delta").mode("overwrite").save("./data/silver_flights")
print("✅ Silver layer cleaned, casted, and saved!")

# 7. Verify the output
df_silver.show(5)

✅ Success! Retrieved 2000 flights.


✅ Bronze layer saved safely!


✅ Silver layer cleaned, casted, and saved!
+------+--------+---------+--------+-------------+--------+----------+---------+
|icao24|callsign|longitude|latitude|baro_altitude|velocity|true_track|on_ground|
+------+--------+---------+--------+-------------+--------+----------+---------+
|39de4f|TVF8939 |    8.417| 41.4058|     10965.18|  223.53|    341.06|    false|
|ab1644|UAL1366 | -78.4682| 38.8383|      6911.34|  198.16|    258.16|    false|
|39de4a|TVF21ZC |   3.0002| 48.7862|       2743.2|  135.38|     67.43|    false|
|39de57|TVF38CS |  -0.0178|  47.939|     10546.08|  242.38|    226.72|    false|
|488284|SPAWE   |  20.8597| 52.2764|       327.66|   33.64|    336.57|    false|
+------+--------+---------+--------+-------------+--------+----------+---------+
only showing top 5 rows

